# Huobi Data Downloader

### Objective:
- Download and save historical price data from the cryptocurrency exchange as CSV files.

In [8]:
# Collecting all imports in one place
import pandas as pd
import requests
import time
import random
import re
import zipfile
import os 

## Function for data downloading

In [11]:
#Function for downloading data from Huobi website
def download_file(data_type='trades',   #Options: index-klines, klines, mark-price-klines, trades.
                  market_type='spot',   #For trades & klines: future, linear-swap, option, spot, swap. For index-klines: linear-swap, swap. For mark-price-klines: future, linear-swap, swap.
                  timeframe='5m',       #For klines: 1min, 5min, 15min, 30min, 60min, 4hour, 1day.
                  date='2024-01-01',    #Format for monthly: 2020-08, for daily: 2020-08-08.
                  ticker='BTCUSDT'):
    
    #Format the final URL string if the data type is candlesticks
    if 'klines' in data_type.lower():
        url = f"https://www.htx.com/data/{data_type}/{market_type}/daily/{ticker}/{timeframe}/{ticker}-{timeframe}-{date}.zip"
    #In all other cases
    else:
        url = f"https://www.htx.com/data/{data_type}/{market_type}/daily/{ticker}/{ticker}-{data_type}-{date}.zip"
    print(url)
    
    #Directory for saving the file
    directory = f"/Users/danielschollenberg/Documents/Трейдинг/Данные/huobi/{market_type}/{data_type}/{ticker}"
    #Create directory if it does not exist
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory {directory} created.")
            
    #Bypassing potential blocking:
    #Add random delay from 0 to 2 seconds before request
    delay = random.uniform(0, 2)
    print(f"Delay before request: {delay:.2f} seconds")
    time.sleep(delay)
    #List of User-Agent headers
    user_agents = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.1 Safari/605.1.15',
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                   'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1',
                   'Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.105 Mobile Safari/537.36']
    #Select random User-Agent header
    headers = {'User-Agent': random.choice(user_agents)}
       
    #Create request to get data in streaming mode
    response = requests.get(url, headers=headers, stream=True)
    #If connection successful (status 200), download and save the file
    if response.status_code == 200:
        filename = directory + f"/{ticker}_{date}.zip"
        with open(filename, 'wb') as f:
            #Write all content directly to the file
            f.write(response.content)  
        print(f"File {filename} successfully downloaded and saved")
    else:
        print(f"Error downloading file: status {response.status_code}, URL: {url}")
        
    #Unpacks zip file into the specified directory  
    def unpack_zip(zip_path, extract_to):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            #Delete zip file after extraction
            os.remove(zip_path)
            print(f"Archive {zip_path} deleted after extraction.")
        except zipfile.BadZipFile:
            print("Error during extraction: file is not a zip archive")
    try:
        
        #Unpack file
        unpack_zip(filename, os.path.dirname(filename))
        print(f"File {filename} successfully extracted.")
    except:
        print('Error during extraction')

In [ ]:
# Example of usage
download_file(data_type='klines', timeframe='1day')